# Diagnostic 0: Cross-Policy Action Diversity — Hopper vs Lift

Compare how much target policies disagree on actions across the two domains.
This is the most direct test of whether guidance-based multi-policy ranking is feasible:
if policies all predict similar actions, no scoring function can distinguish them.

**Metrics:**
1. Per-state cross-policy action std (normalized by dataset action std)
2. Pairwise mean L2 distance between policy actions (normalized)
3. Action agreement rate (fraction of states where all policies agree within threshold)

In [ ]:
import sys, os
os.environ["D4RL_SUPPRESS_IMPORT_ERROR"] = "1"
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
sys.path.insert(0, os.path.join(os.getcwd(), "../../third_party/sope"))
sys.path.insert(0, os.path.join(os.getcwd(), "../.."))

import torch
import numpy as np
import h5py
from itertools import combinations
%matplotlib inline
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
np.random.seed(42)
torch.manual_seed(42)
print(f"Device: {DEVICE}")

## Part 1: Hopper D4RL — 11 SAC Policies

In [ ]:
import gym, d4rl
from opelab.core.policy import D4RLSACPolicy, D4RLPolicy

SOPE_ROOT = os.path.join(os.getcwd(), "../../third_party/sope")

# Load Hopper dataset
env = gym.make("hopper-medium-v2")
dataset = env.get_dataset()
hopper_obs = dataset["observations"]
hopper_actions = dataset["actions"]
HOPPER_STATE_DIM, HOPPER_ACTION_DIM = 11, 3
print(f"Hopper dataset: {hopper_obs.shape[0]} transitions, state_dim={HOPPER_STATE_DIM}, action_dim={HOPPER_ACTION_DIM}")

# Dataset action statistics for normalization
hopper_action_std = np.std(hopper_actions, axis=0)
print(f"Hopper action std per dim: {hopper_action_std}")

# Load 11 SAC policies
policy_dir = os.path.join(SOPE_ROOT, "opelab/examples/d4rl/policy/hopper/dope")
policy_paths = sorted(
    [os.path.join(policy_dir, f) for f in os.listdir(policy_dir) if f.endswith(".pkl")],
    key=lambda p: int(os.path.basename(p).replace(".pkl", ""))
)
hopper_policies = {int(os.path.basename(p).replace(".pkl","")): D4RLSACPolicy(p).to(DEVICE) for p in policy_paths}
print(f"Loaded {len(hopper_policies)} Hopper target policies")

In [ ]:
# Sample 500 random states from Hopper dataset
N_STATES_HOPPER = 500
idx = np.random.choice(len(hopper_obs), N_STATES_HOPPER, replace=False)
sampled_states_hopper = torch.tensor(hopper_obs[idx], dtype=torch.float32, device=DEVICE)
sampled_actions_hopper = hopper_actions[idx]  # for reference

# Get deterministic action from each policy at each state
hopper_policy_actions = {}  # {policy_id: (N_STATES, ACTION_DIM)}
for pi, policy in sorted(hopper_policies.items()):
    with torch.no_grad():
        acts = policy.sample_tensor(sampled_states_hopper, deterministic=True)
    hopper_policy_actions[pi] = acts.cpu().numpy()
    print(f"  Policy {pi}: mean action = {acts.mean(0).cpu().numpy()}")

print(f"\nCollected actions from {len(hopper_policy_actions)} policies at {N_STATES_HOPPER} states")

## Part 2: Lift — DiffusionPolicyUNet Policies (varying demo count)

In [ ]:
from src.latent_sope.robomimic_interface.checkpoints import (
    load_checkpoint, build_rollout_policy_from_checkpoint
)

CKPT_BASE = "/home1/reishuen/latent_sope/third_party/robomimic/diffusion_policy_trained_models"
LIFT_POLICIES = {
    "10demos_e40":  (f"{CKPT_BASE}/lift_diffusion_10demos/20260311115828", "models/model_epoch_40.pth"),
    "50demos_e40":  (f"{CKPT_BASE}/lift_diffusion_50demos/20260311134204", "models/model_epoch_40.pth"),
    "100demos_e40": (f"{CKPT_BASE}/lift_diffusion_100demos/20260311135551", "models/model_epoch_40.pth"),
    "200demos_e40": (f"{CKPT_BASE}/lift_diffusion_200demos/20260311141036", "models/model_epoch_40.pth"),
}

# Also include different epochs from the same checkpoint to test epoch diversity
LIFT_POLICIES["200demos_e10"] = (f"{CKPT_BASE}/lift_diffusion_200demos/20260311141036", "models/model_epoch_10.pth")
LIFT_POLICIES["200demos_e20"] = (f"{CKPT_BASE}/lift_diffusion_200demos/20260311141036", "models/model_epoch_20.pth")

OBS_KEYS = ["object", "robot0_eef_pos", "robot0_eef_quat", "robot0_gripper_qpos"]
LIFT_STATE_DIM, LIFT_ACTION_DIM = 19, 7

# Load rollout policies
lift_rollout_policies = {}
for name, (run_dir, ckpt_rel) in LIFT_POLICIES.items():
    print(f"Loading {name}...")
    ckpt = load_checkpoint(run_dir=run_dir, ckpt_path=ckpt_rel)
    policy = build_rollout_policy_from_checkpoint(ckpt, device=DEVICE, verbose=False)
    lift_rollout_policies[name] = policy
    print(f"  Loaded: {name}")

print(f"\nLoaded {len(lift_rollout_policies)} Lift policies")

In [ ]:
# Sample states from Lift demo dataset
DEMO_H5_PATH = "/home1/reishuen/latent_sope/third_party/robomimic/datasets/lift/ph/low_dim_v15.hdf5"
N_STATES_LIFT = 200

# Load demo observations
with h5py.File(DEMO_H5_PATH, "r") as f:
    demo_keys = sorted(list(f["data"].keys()))
    all_obs = {k: [] for k in OBS_KEYS}
    all_actions = []
    for dk in demo_keys:
        for k in OBS_KEYS:
            all_obs[k].append(f["data"][dk]["obs"][k][:])
        all_actions.append(f["data"][dk]["actions"][:])
    for k in OBS_KEYS:
        all_obs[k] = np.concatenate(all_obs[k], axis=0)
    all_actions_np = np.concatenate(all_actions, axis=0)

total_steps = all_obs[OBS_KEYS[0]].shape[0]
print(f"Demo dataset: {total_steps} transitions from {len(demo_keys)} demos")

# Dataset action statistics for normalization
lift_action_std = np.std(all_actions_np, axis=0)
print(f"Lift action std per dim: {lift_action_std}")

# Sample random indices
sample_idx = np.random.choice(total_steps, N_STATES_LIFT, replace=False)
sampled_obs_dicts = [{k: all_obs[k][i] for k in OBS_KEYS} for i in sample_idx]
print(f"Sampled {N_STATES_LIFT} states")

In [ ]:
# Query each Lift policy at each sampled state
# DiffusionPolicyUNet is stochastic — sample once per state (cross-policy variance should dominate)
#
# IMPORTANT: DiffusionPolicyUNet has observation_horizon=2, meaning the noise network
# expects 2 stacked frames of observations. RolloutPolicy doesn't handle this automatically.
# We pass obs with shape (2, D) per key by repeating the single observation twice.
# This is equivalent to "no history" — both frames see the same state.

lift_policy_actions = {}  # {name: (N_STATES, ACTION_DIM)}

for name, policy in sorted(lift_rollout_policies.items()):
    print(f"Querying {name} at {N_STATES_LIFT} states...")
    actions_list = []
    for i, obs_dict in enumerate(sampled_obs_dicts):
        policy.start_episode()
        # Stack 2 copies of each obs key to satisfy observation_horizon=2
        obs_stacked = {k: np.stack([v, v], axis=0).astype(np.float32) for k, v in obs_dict.items()}
        act = policy(obs_stacked)  # returns numpy (action_dim,)
        actions_list.append(act)
        if (i + 1) % 50 == 0:
            print(f"    {i+1}/{N_STATES_LIFT}")
    lift_policy_actions[name] = np.stack(actions_list, axis=0)
    print(f"  Done. Mean action: {lift_policy_actions[name].mean(0)}")

print(f"\nCollected actions from {len(lift_policy_actions)} Lift policies at {N_STATES_LIFT} states")

## Metric Computation

In [ ]:
def compute_diversity_metrics(policy_actions, action_std, domain_name):
    """
    policy_actions: {name: (N, action_dim)} — actions from each policy at N shared states
    action_std: (action_dim,) — dataset action std for normalization
    Returns dict of metrics.
    """
    names = sorted(policy_actions.keys())
    N = next(iter(policy_actions.values())).shape[0]
    action_dim = next(iter(policy_actions.values())).shape[1]
    n_policies = len(names)
    
    # Stack: (n_policies, N, action_dim)
    stacked = np.stack([policy_actions[n] for n in names], axis=0)
    
    # --- Metric 1: Per-state cross-policy action std (normalized) ---
    # At each state, compute std across policies, then normalize by dataset std
    per_state_std = np.std(stacked, axis=0)  # (N, action_dim)
    normalized_std = per_state_std / action_std[None, :]  # (N, action_dim)
    mean_normalized_std = np.mean(normalized_std)  # scalar
    per_dim_normalized_std = np.mean(normalized_std, axis=0)  # (action_dim,)
    
    # --- Metric 2: Pairwise mean L2 distance (normalized) ---
    pairwise_dists = []
    for i, j in combinations(range(n_policies), 2):
        diff = (stacked[i] - stacked[j]) / action_std[None, :]  # normalized
        l2 = np.sqrt(np.sum(diff**2, axis=-1))  # (N,)
        pairwise_dists.append(np.mean(l2))
    mean_pairwise_dist = np.mean(pairwise_dists)
    min_pairwise_dist = np.min(pairwise_dists)
    max_pairwise_dist = np.max(pairwise_dists)
    
    # Pairwise matrix for display
    pairwise_matrix = np.zeros((n_policies, n_policies))
    idx = 0
    for i, j in combinations(range(n_policies), 2):
        pairwise_matrix[i, j] = pairwise_dists[idx]
        pairwise_matrix[j, i] = pairwise_dists[idx]
        idx += 1
    
    # --- Metric 3: Action agreement rate ---
    # Fraction of states where ALL policies agree within 0.1 * action_std
    threshold = 0.1  # in normalized space
    per_state_range = np.max(stacked, axis=0) - np.min(stacked, axis=0)  # (N, action_dim)
    normalized_range = per_state_range / action_std[None, :]  # (N, action_dim)
    all_agree = np.all(normalized_range < threshold, axis=-1)  # (N,)
    agreement_rate = np.mean(all_agree)
    
    # Also compute per-dim agreement
    per_dim_agree = np.mean(normalized_range < threshold, axis=0)  # (action_dim,)
    
    results = {
        "domain": domain_name,
        "n_policies": n_policies,
        "n_states": N,
        "action_dim": action_dim,
        "mean_normalized_std": mean_normalized_std,
        "per_dim_normalized_std": per_dim_normalized_std,
        "mean_pairwise_dist": mean_pairwise_dist,
        "min_pairwise_dist": min_pairwise_dist,
        "max_pairwise_dist": max_pairwise_dist,
        "pairwise_matrix": pairwise_matrix,
        "policy_names": names,
        "agreement_rate": agreement_rate,
        "per_dim_agree": per_dim_agree,
    }
    
    # Print summary
    print(f"\n{'='*60}")
    print(f"{domain_name} Action Diversity Metrics")
    print(f"{'='*60}")
    print(f"Policies: {n_policies}, States: {N}, Action dim: {action_dim}")
    print(f"")
    print(f"Metric 1: Mean normalized cross-policy std = {mean_normalized_std:.4f}")
    print(f"  Per-dim: {per_dim_normalized_std}")
    print(f"")
    print(f"Metric 2: Mean pairwise L2 distance (normalized) = {mean_pairwise_dist:.4f}")
    print(f"  Range: [{min_pairwise_dist:.4f}, {max_pairwise_dist:.4f}]")
    print(f"")
    print(f"Metric 3: Agreement rate (all within 0.1σ) = {agreement_rate:.4f} ({agreement_rate*100:.1f}%)")
    print(f"  Per-dim agreement: {per_dim_agree}")
    print(f"")
    print(f"Pairwise distance matrix:")
    header = "         " + " ".join(f"{n:>10s}" for n in names)
    print(header)
    for i, ni in enumerate(names):
        row = f"{ni:>8s} " + " ".join(f"{pairwise_matrix[i,j]:>10.4f}" for j in range(n_policies))
        print(row)
    
    return results

In [ ]:
hopper_results = compute_diversity_metrics(hopper_policy_actions, hopper_action_std, "Hopper D4RL")
lift_results = compute_diversity_metrics(lift_policy_actions, lift_action_std, "Lift Robomimic")

## Side-by-Side Comparison

In [ ]:
print(f"\n{'='*70}")
print(f"SIDE-BY-SIDE COMPARISON: Hopper vs Lift")
print(f"{'='*70}")
print(f"")
print(f"{'Metric':<45s} {'Hopper':>10s} {'Lift':>10s} {'Ratio':>10s}")
print(f"{'-'*75}")

h, l = hopper_results, lift_results
print(f"{'Num policies':<45s} {h['n_policies']:>10d} {l['n_policies']:>10d}")
print(f"{'Num states sampled':<45s} {h['n_states']:>10d} {l['n_states']:>10d}")
print(f"{'Action dim':<45s} {h['action_dim']:>10d} {l['action_dim']:>10d}")
print(f"")

ratio_std = h['mean_normalized_std'] / max(l['mean_normalized_std'], 1e-8)
print(f"{'Mean normalized cross-policy std':<45s} {h['mean_normalized_std']:>10.4f} {l['mean_normalized_std']:>10.4f} {ratio_std:>10.2f}x")

ratio_dist = h['mean_pairwise_dist'] / max(l['mean_pairwise_dist'], 1e-8)
print(f"{'Mean pairwise L2 distance (normalized)':<45s} {h['mean_pairwise_dist']:>10.4f} {l['mean_pairwise_dist']:>10.4f} {ratio_dist:>10.2f}x")

print(f"{'Agreement rate (all within 0.1σ)':<45s} {h['agreement_rate']*100:>9.1f}% {l['agreement_rate']*100:>9.1f}%")
print(f"")
print(f"Higher std & distance = more diverse policies = guidance can distinguish them.")
print(f"Higher agreement rate = policies are more similar = guidance is blind.")

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Bar chart: mean normalized std per domain
ax = axes[0]
bars = ax.bar(["Hopper", "Lift"], 
              [hopper_results['mean_normalized_std'], lift_results['mean_normalized_std']],
              color=["steelblue", "coral"], alpha=0.8)
ax.set_ylabel("Mean Normalized Cross-Policy Std")
ax.set_title("Action Diversity: Cross-Policy Std")
for bar, val in zip(bars, [hopper_results['mean_normalized_std'], lift_results['mean_normalized_std']]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f"{val:.4f}",
            ha='center', va='bottom', fontsize=11)

# 2. Pairwise distance distributions
ax = axes[1]
# Extract upper triangle values
h_pw = hopper_results['pairwise_matrix']
l_pw = lift_results['pairwise_matrix']
h_vals = h_pw[np.triu_indices_from(h_pw, k=1)]
l_vals = l_pw[np.triu_indices_from(l_pw, k=1)]
ax.hist(h_vals, bins=20, alpha=0.6, label=f"Hopper (n={len(h_vals)})", color="steelblue")
ax.hist(l_vals, bins=20, alpha=0.6, label=f"Lift (n={len(l_vals)})", color="coral")
ax.set_xlabel("Pairwise L2 Distance (normalized)")
ax.set_ylabel("Count")
ax.set_title("Distribution of Pairwise Distances")
ax.legend()

# 3. Per-dimension normalized std comparison
ax = axes[2]
x_h = np.arange(HOPPER_ACTION_DIM)
x_l = np.arange(LIFT_ACTION_DIM)
ax.bar(x_h - 0.2, hopper_results['per_dim_normalized_std'], 0.4, 
       label="Hopper", color="steelblue", alpha=0.8)
ax.bar(x_l + 0.2, lift_results['per_dim_normalized_std'][:LIFT_ACTION_DIM], 0.4,
       label="Lift", color="coral", alpha=0.8)
ax.set_xlabel("Action Dimension")
ax.set_ylabel("Normalized Cross-Policy Std")
ax.set_title("Per-Dimension Action Diversity")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Pairwise distance heatmaps
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, res, title in [(axes[0], hopper_results, "Hopper"), (axes[1], lift_results, "Lift")]:
    im = ax.imshow(res['pairwise_matrix'], cmap='YlOrRd', aspect='auto')
    ax.set_xticks(range(res['n_policies']))
    ax.set_yticks(range(res['n_policies']))
    ax.set_xticklabels(res['policy_names'], rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(res['policy_names'], fontsize=8)
    ax.set_title(f"{title}: Pairwise L2 Distance (normalized)")
    plt.colorbar(im, ax=ax)
    # Annotate cells
    for i in range(res['n_policies']):
        for j in range(res['n_policies']):
            if i != j:
                ax.text(j, i, f"{res['pairwise_matrix'][i,j]:.2f}",
                       ha='center', va='center', fontsize=7)

plt.tight_layout()
plt.show()

## Bonus: Within-Policy Stochasticity (Lift only)

DiffusionPolicyUNet is stochastic — how much does a single policy vary across samples at the same state?
If within-policy variance > cross-policy variance, the diversity signal is drowned out.

In [ ]:
# Pick one Lift policy (200demos_e40) and sample K=5 times at 50 states
K_SAMPLES = 5
N_STATES_STOCH = 50
test_policy_name = "200demos_e40"
test_policy = lift_rollout_policies[test_policy_name]

stoch_actions = np.zeros((K_SAMPLES, N_STATES_STOCH, LIFT_ACTION_DIM))
for k in range(K_SAMPLES):
    for i in range(N_STATES_STOCH):
        test_policy.start_episode()
        obs_stacked = {key: np.stack([v, v], axis=0).astype(np.float32) for key, v in sampled_obs_dicts[i].items()}
        act = test_policy(obs_stacked)
        stoch_actions[k, i] = act
    print(f"  Sample {k+1}/{K_SAMPLES} done")

# Within-policy std (across K samples at same state)
within_std = np.std(stoch_actions, axis=0)  # (N_STATES_STOCH, action_dim)
within_normalized = within_std / lift_action_std[None, :]
mean_within = np.mean(within_normalized)

# Cross-policy std at same states (from earlier data)
cross_at_50 = np.stack([lift_policy_actions[n][:N_STATES_STOCH] for n in sorted(lift_policy_actions.keys())], axis=0)
cross_std_50 = np.std(cross_at_50, axis=0) / lift_action_std[None, :]
mean_cross = np.mean(cross_std_50)

print(f"\nWithin-policy stochasticity ({test_policy_name}, K={K_SAMPLES}):")
print(f"  Mean normalized within-policy std:  {mean_within:.4f}")
print(f"  Mean normalized cross-policy std:   {mean_cross:.4f}")
print(f"  Ratio (cross/within):               {mean_cross/max(mean_within, 1e-8):.2f}x")
print(f"")
if mean_cross < mean_within:
    print(f"  WARNING: Cross-policy variance < within-policy stochasticity!")
    print(f"  The diversity signal is drowned out by sampling noise.")
else:
    print(f"  Cross-policy variance dominates — diversity signal is detectable.")

In [ ]:
# Final summary for results logging
print("\n" + "="*70)
print("FINAL SUMMARY FOR RESULTS LOG")
print("="*70)
print(f"")
print(f"| Metric | Hopper D4RL | Lift Robomimic | Ratio (H/L) |")
print(f"|--------|-------------|----------------|-------------|")
print(f"| Num policies | {h['n_policies']} | {l['n_policies']} | — |")
print(f"| Num states | {h['n_states']} | {l['n_states']} | — |")
print(f"| Action dim | {h['action_dim']} | {l['action_dim']} | — |")
print(f"| **Mean normalized cross-policy std** | **{h['mean_normalized_std']:.4f}** | **{l['mean_normalized_std']:.4f}** | **{ratio_std:.2f}x** |")
print(f"| **Mean pairwise L2 (normalized)** | **{h['mean_pairwise_dist']:.4f}** | **{l['mean_pairwise_dist']:.4f}** | **{ratio_dist:.2f}x** |")
print(f"| Agreement rate (all within 0.1σ) | {h['agreement_rate']*100:.1f}% | {l['agreement_rate']*100:.1f}% | — |")
print(f"| Within-policy std (Lift only) | — | {mean_within:.4f} | — |")
print(f"| Cross/within ratio (Lift only) | — | {mean_cross/max(mean_within, 1e-8):.2f}x | — |")
print(f"")
print("Done.")